# Data inspection: dicarlo.Rajalingham2018.public


The files are pulled directly from the DiCarlo lab's S3 bucket and hash-verified.

In [ ]:
import hashlib
import os
import requests

CACHE_DIR = os.path.expanduser("~/.brainio_manual")
os.makedirs(CACHE_DIR, exist_ok=True)

# URLs and sha1 hashes copied from brainio_collection's own lookup.csv
FILES = {
    "assembly": (
        "https://s3.amazonaws.com/brainio.dicarlo/assy_dicarlo_Rajalingham2018_public.nc",
        "34c6a8b6f7c523589c1861e4123232e5f7c7df4c",
    ),
    "stimuli_csv": (
        "https://s3.amazonaws.com/brainio.dicarlo/image_dicarlo_objectome_public.csv",
        "47884e17106a3be471d6481279cab33889b80850",
    ),
    "stimuli_zip": (
        "https://s3.amazonaws.com/brainio.dicarlo/image_dicarlo_objectome_public.zip",
        "064f2955f98e63867755fee2e3ead8cddf6bfab8",
    ),
}

paths = {}
for name, (url, expected_sha1) in FILES.items():
    local_path = os.path.join(CACHE_DIR, os.path.basename(url))
    if not os.path.exists(local_path):
        open(local_path, "wb").write(requests.get(url, timeout=60).content)

    actual_sha1 = hashlib.sha1(open(local_path, "rb").read()).hexdigest()
    assert actual_sha1 == expected_sha1, f"sha1 mismatch for {name}"
    paths[name] = local_path
    print(f"{name}: fetched and verified OK")

All three files downloaded and SHA-1-verified against brainio's own lookup table (confirmed it is a bit-for-bit the same files brain-score itself would fetch). Files downloaded:<br>
1. assembly - the raw behavioural data<br>
2. stimulus zip - set of image used as stimuli during behavioural trials<br>
3. stimulus CSV - image metadata (which category each image belongs to)


## 1. The behavioural dataset (assembly)

In [ ]:
import xarray as xr
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

assembly = xr.open_dataarray(paths["assembly"])
print("n trials:", len(assembly))
print("one row per trial?", assembly.dims == ("presentation",))

In [ ]:
# Build a plain dataframe of everything, then check the columns mean what their names suggest.
df = pd.DataFrame({c: assembly.coords[c].values for c in assembly.coords})
df["values"] = assembly.values
print(df.shape)

from IPython.display import display
display(df.head(5))

print("dtypes:\n", df.dtypes)
print()
print("n unique image_id:", df["image_id"].nunique())
print("n unique WorkerID (subjects):", df["WorkerID"].nunique())
print("n unique AssignmentID:", df["AssignmentID"].nunique())
print("n unique sample_obj (categories?):", df["sample_obj"].nunique())
print("n unique dist_obj:", df["dist_obj"].nunique())
print()
print("sample_obj == truth always?", (df["sample_obj"] == df["truth"]).all())
print("choice in {sample_obj, dist_obj} always?",
      ((df["choice"] == df["sample_obj"]) | (df["choice"] == df["dist_obj"])).all())

2,160 unique images and 24 categories confirmed.<br>
sample_obj (the target category) and truth (ground truth) are the same throughout the table as it should be the case.<br>
The recorded choice should match one of the two offered choices - target or distractor. In the next cell it was found that 2 out of 585,511 do not match.

In [ ]:
# choice not always in {sample_obj, dist_obj} - look at the mismatches
mismatch = df[~((df["choice"] == df["sample_obj"]) | (df["choice"] == df["dist_obj"]))]
print("n mismatched rows:", len(mismatch), "out of", len(df))
mismatch[["sample_obj", "dist_obj", "choice", "truth"]]

Only 2 rows out of 585,511 have a `choice` that matches neither the sample nor distractor object shown on that trial.<br>
These 2 rows are EXCLUDED.

## 2. The stimulus set: images, categories, tested/untested split

In [ ]:
stim_df = pd.read_csv(paths["stimuli_csv"])
print("shape:", stim_df.shape)
print("columns:", list(stim_df.columns))
stim_df.head(10)

**Stimulus metadata file** 2,160 rows, one per image, giving each a category label (`image_label`), a content-hash ID, and a filename - the ground-truth category list, independent of anything a human observer did.

In [ ]:
# Category balance + cross-check against the trial data's image_ids
counts = stim_df["image_label"].value_counts()
print("n unique categories in stimulus set:", stim_df["image_label"].nunique())
print("all categories have equal image counts?", counts.nunique() == 1, f"({counts.iloc[0]} each)")
print()
print("n images in stimulus set:", stim_df["image_id"].nunique())
print("n images with >=1 trial:", df["image_id"].nunique())

**Key finding:** `.public` is exactly 2160 images (24 categories x 90), and every image has trial data (no untested image inside `.public`). The original objectome scope is 2400 images. `2400 - 2160 = 240 = 24 x 10`, which is exactly the "240 images with the most trials" that the Brain-Score paper says it scored on and is held private.

This project therefore uses the public 2,160-image publicly released and will sudsample from this dataset a new 240 held-out subset for testing.

## 3. Trial structure: how many trials per (image, target, distractor) combination?

In [ ]:
# Trials per (image_id, sample_obj, dist_obj) combination
trial_counts = df.groupby(["image_id", "sample_obj", "dist_obj"]).size()
print("n distinct (image, target, distractor) combinations:", len(trial_counts))
print(f"trials per combination - min: {trial_counts.min()}, median: {trial_counts.median()}, max: {trial_counts.max()}")
print()
per_image = df.groupby("image_id").size()
print(f"trials per image_id (summed over all distractors) - min: {per_image.min()}, median: {per_image.median()}, max: {per_image.max()}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(trial_counts, bins=30)
plt.xlabel("trials per (image, target, distractor) combination")
plt.ylabel("number of combinations")
plt.title("Trial-count distribution across combinations")
plt.tight_layout()
plt.show()

**Trials distribution across combinations**<br>
Distinct (image, target, distractor) combinations that got tested at least once: 49,492 (also the number of folders)<br>
- min: 1<br>
- median: 10<br>
- max: 64<br>

## 4. Image format: resolution, colour mode, masking

In [ ]:
import zipfile
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

IMG_DIR = os.path.join(CACHE_DIR, "objectome_public_images")
os.makedirs(IMG_DIR, exist_ok=True)
with zipfile.ZipFile(paths["stimuli_zip"]) as zf:
    names = zf.namelist()
    print("n entries in zip:", len(names))
    sample_names = [n for n in names if n.endswith(".png")][:5]
    for n in sample_names:
        zf.extract(n, IMG_DIR)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, n in zip(axes, sample_names):
    im = Image.open(os.path.join(IMG_DIR, n))
    ax.imshow(im, cmap="gray")
    ax.set_title(n[:8], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Greyscale check (R==G==B) 
for n in sample_names:
    a = np.array(Image.open(os.path.join(IMG_DIR, n)))
    is_grey = np.array_equal(a[..., 0], a[..., 1]) and np.array_equal(a[..., 1], a[..., 2])
    print(f"{n[:8]}: mode/size={a.shape}, greyscale={is_grey}")


## Summary

**Public/private status:**
- `dicarlo.Rajalingham2018.public` = 2160 images, 24 categories x 90 images. Every image already has trial data (585,511 trials total, 137-723 trials per image).
- The classic objectome corpus is 2400 images; `2400 - 2160 = 240 = 24 x 10`, matching exactly the "240 images with the most trials" that Brain-Score's paper says it scored on.
- **Conclusion: with public access only, there is no untested/held-out image inside what can be downloaded.** All 2160 public images are "tested" and have human trial data.

**Trial-level data, not aggregated:** the assembly is a flat `(presentation,)` array - one row per individual 2AFC trial, with `image_id`, `sample_obj` (target, ==`truth` always), `dist_obj` (distractor), `choice` (what was picked), `WorkerID` (subject), `AssignmentID`.

**Data quality note:** 2 out of 585,511 trials (0.0003%) have a `choice` that matches neither the target nor the distractor shown - excluded.
**Images:** 224x224 RGB-encoded PNGs that are genuinely greyscale (R=G=B). Natural-scene backgrounds with a rendered object composited in (see figures above).
**Subjects:** `WorkerID` (1391 unique); `AssignmentID` (8185 unique) identifies a single sitting/session.